In [1]:
from dataclasses import dataclass
import math
import random
import numpy as np
import os
from pathlib import Path
from PIL import Image
from PIL.ExifTags import TAGS
from collections import Counter
import tqdm
import torch
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, fcluster
import time
from datetime import datetime
import exiftool


In [2]:
DATA_ROOT = '/home/slavik/e202602_eclipse/data'
BRIGHTNESS_MIN = 0.0
BRIGHTNESS_MAX = 1.0
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [3]:

@dataclass
class ImageInfo:
    path: Path
    width: int
    height: int
    avg_brightness: float
    timestamp: float
    exposure_time: float  # seconds



In [4]:
def get_image_timestamp(img: Image.Image, img_path: Path) -> float:
    """
    Extract timestamp from image EXIF data.
    Returns timestamp as float (Unix timestamp format like time.time()).
    Raises ValueError if EXIF data is missing or format is invalid.
    """
    exif = img.getexif()
    if exif is None:
        raise ValueError(f"No EXIF data found in {img_path}")
    
    # Try DateTimeOriginal (36867) first, fallback to DateTime (306)
    datetime_original_tag = 36867
    datetime_tag = 306
    if datetime_original_tag in exif:
        datetime_str = exif[datetime_original_tag]
    elif datetime_tag in exif:
        datetime_str = exif[datetime_tag]
    else:
        raise ValueError(f"Neither DateTimeOriginal (36867) nor DateTime (306) found in EXIF data for {img_path}")
    
    if not isinstance(datetime_str, str):
        raise ValueError(f"DateTime value is not a string in {img_path}: {datetime_str}")
    
    # Parse the datetime string (format: "YYYY:MM:DD HH:MM:SS")
    # Check that it has second-level precision (format should be exactly "YYYY:MM:DD HH:MM:SS")
    expected_format = "%Y:%m:%d %H:%M:%S"
    if len(datetime_str) != len("YYYY:MM:DD HH:MM:SS") or datetime_str.count(':') != 4:
        raise ValueError(f"DateTime format is not precise to seconds in {img_path}: {datetime_str}")
    
    try:
        dt = datetime.strptime(datetime_str, expected_format)
        timestamp = dt.timestamp()
    except ValueError as e:
        raise ValueError(f"Failed to parse DateTime '{datetime_str}' in {img_path}: {e}")
    
    return timestamp


def get_image_exposure_time(img_path: Path) -> float:
    """
    Extract exposure time in seconds from image EXIF via exiftool.
    Raises ValueError if EXIF:ExposureTime is missing.
    """
    with exiftool.ExifToolHelper() as et:
        metadata = et.get_metadata(str(img_path))[0]
    raw = metadata.get('EXIF:ExposureTime')
    if raw is None:
        raise ValueError(f"EXIF:ExposureTime not found in {img_path}")
    if isinstance(raw, (int, float)):
        return float(raw)
    s = str(raw).strip()
    if '/' in s:
        num, den = s.split('/', 1)
        return float(num) / float(den) if float(den) else 0.0
    return float(s)


def get_image_infos():
    jpg_files = list(Path(DATA_ROOT).rglob('*.jpg')) + list(Path(DATA_ROOT).rglob('*.JPG'))
    image_infos = []
    for jpg_file in tqdm.tqdm(jpg_files, desc="First scan of images"):
        with Image.open(jpg_file) as img:
            width, height = img.size
            avg_brightness = np.array(img).astype(np.float32).mean() / 255.0
            if BRIGHTNESS_MIN <= avg_brightness <= BRIGHTNESS_MAX:
                timestamp = get_image_timestamp(img, jpg_file)
                exposure_time = get_image_exposure_time(jpg_file)
                image_infos.append(ImageInfo(path=jpg_file, width=width, height=height, avg_brightness=avg_brightness, timestamp=timestamp, exposure_time=exposure_time))
    assert len(image_infos) > 0
    for ii in image_infos:
        assert ii.width == image_infos[0].width
        assert ii.height == image_infos[0].height
    image_infos.sort(key=lambda x: x.avg_brightness)
    return image_infos

In [5]:
N_SECTORS = 360
N_TRIPLETS = 1024
N_CLUSTER = 256
DEBUG_RADIUS_PX = 4
REFINE_ITERATIONS = 3
MIN_TRIPLET_DEGREES = 30


def _indices_within_degrees(center: int, deg: int) -> set:
    """Sector indices within deg degrees of center (wrap-around)."""
    return {(center + d) % N_SECTORS for d in range(-(deg - 1), deg)}


def sample_triplet_indices(n_pts: int, min_degrees: int = MIN_TRIPLET_DEGREES) -> tuple[int, int, int]:
    """
    Sample three distinct sector indices such that each pair is at least min_degrees apart.
    Falls back to unrestricted random triplet if not enough spread is available.
    """
    available = set(range(n_pts))
    a = random.sample(list(available), 1)[0]
    available -= _indices_within_degrees(a, min_degrees) & available
    assert len(available) >= 2
    b = random.sample(list(available), 1)[0]
    available -= _indices_within_degrees(b, min_degrees) & available
    assert len(available) >= 1
    c = random.sample(list(available), 1)[0]
    return (a, b, c)


def refine_moon(img: torch.Tensor, center_i: float, center_j: float) -> tuple[float, float, float]:
    """
    Refine moon center from image and current center (i, j).
    img: (H, W, 3) float32 [0,1] on GPU.
    Returns (i, j, radius) as tuple of (float, float, float).
    """
    assert img.ndim == 3 and img.shape[2] == 3
    H, W = img.shape[0], img.shape[1]
    dev = img.device
    img_size = float(max(H, W))

    gray = img.mean(dim=2)
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32, device=dev).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32, device=dev).view(1, 1, 3, 3)
    g = gray.unsqueeze(0).unsqueeze(0)
    grad_x = torch.nn.functional.conv2d(g, sobel_x, padding=1).squeeze()
    grad_y = torch.nn.functional.conv2d(g, sobel_y, padding=1).squeeze()

    dy = torch.arange(H, device=dev, dtype=torch.float32).view(-1, 1) - center_i
    dx = torch.arange(W, device=dev, dtype=torch.float32).view(1, -1) - center_j
    norm = torch.sqrt(dx * dx + dy * dy).clamp(min=1e-6)
    u_x = dx / norm
    u_y = dy / norm

    angle = torch.atan2(dy, dx)
    sector_id = (torch.floor((angle + math.pi) / (2 * math.pi) * N_SECTORS).long() % N_SECTORS)

    dot_product = grad_x * u_x + grad_y * u_y
    dot_product_flat = dot_product.reshape(-1)
    sector_flat = sector_id.reshape(-1)
    W_t = W

    points_list = []
    for s in range(N_SECTORS):
        mask = sector_flat == s
        if mask.any():
            masked = torch.where(mask, dot_product_flat, torch.tensor(-1e9, device=dev, dtype=torch.float32))
            idx = masked.argmax().item()
            i, j = idx // W_t, idx % W_t
            points_list.append((i, j))

    n_pts = len(points_list)
    if n_pts < 3:
        return (center_i, center_j, 0.0)

    def circumcenter(i1, j1, i2, j2, i3, j3):
        x1, y1, x2, y2, x3, y3 = float(j1), float(i1), float(j2), float(i2), float(j3), float(i3)
        D = 2.0 * (x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2))
        if abs(D) < 1e-10:
            return None
        ox = ((x1 * x1 + y1 * y1) * (y2 - y3) + (x2 * x2 + y2 * y2) * (y3 - y1) + (x3 * x3 + y3 * y3) * (y1 - y2)) / D
        oy = ((x1 * x1 + y1 * y1) * (x3 - x2) + (x2 * x2 + y2 * y2) * (x1 - x3) + (x3 * x3 + y3 * y3) * (x2 - x1)) / D
        oi, oj = oy, ox
        d1 = math.hypot(i1 - oi, j1 - oj)
        d2 = math.hypot(i2 - oi, j2 - oj)
        d3 = math.hypot(i3 - oi, j3 - oj)
        if d1 > img_size or d2 > img_size or d3 > img_size:
            return None
        # Compute radius as average of distances from circumcenter to the 3 points
        radius = (d1 + d2 + d3) / 3.0
        return (oi, oj, radius)

    circumcenters = []
    radii = []
    while len(circumcenters) < N_TRIPLETS:
        a, b, c = sample_triplet_indices(n_pts)
        i1, j1 = points_list[a]
        i2, j2 = points_list[b]
        i3, j3 = points_list[c]
        cc_result = circumcenter(i1, j1, i2, j2, i3, j3)
        if cc_result is not None:
            oi, oj, radius = cc_result
            circumcenters.append((oi, oj))
            radii.append(radius)

    pts = np.array(circumcenters, dtype=np.float64)
    Z = linkage(pts, method="complete")
    t_lo, t_hi = 0.0, float(Z[-1, 2])
    for _ in range(60):
        t = (t_lo + t_hi) / 2
        labels = fcluster(Z, t, criterion="distance")
        sizes = np.bincount(labels)
        max_size = int(sizes.max())
        if max_size >= N_CLUSTER:
            t_hi = t
        else:
            t_lo = t
    labels = fcluster(Z, t_hi, criterion="distance")
    sizes = np.bincount(labels)
    which = int(np.argmax(sizes))
    cluster_mask = labels == which
    cluster_pts = pts[cluster_mask]
    cluster_radii = np.array(radii)[cluster_mask]
    ci = float(cluster_pts[:, 0].mean())
    cj = float(cluster_pts[:, 1].mean())
    radius = float(cluster_radii.mean())
    return (ci, cj, radius)


def find_moon(img: torch.Tensor, i0: float, j0: float) -> tuple[float, float, float]:
    """
    Find moon center by iteratively refining from image center.
    img: (H, W, 3) float32 [0,1] on GPU. Returns (i, j, radius) as tuple of (float, float, float).
    """
    assert img.ndim == 3 and img.shape[2] == 3
    center_i, center_j = i0, j0
    radius = 0.0
    for _ in range(REFINE_ITERATIONS):
        center_i, center_j, radius = refine_moon(img, center_i, center_j)

    if False:
        img_np = img.cpu().numpy()
        H, W = img_np.shape[0], img_np.shape[1]
        
        # Calculate crop size: 2.2 * radius (10% padding on each side)
        half_crop = int(round(1.1 * radius))
        
        # Calculate crop bounds (square crop centered on moon)
        i_min = int(round(center_i - half_crop))
        i_max = int(round(center_i + half_crop))
        j_min = int(round(center_j - half_crop))
        j_max = int(round(center_j + half_crop))
        
        # Check bounds and throw exception if out of bounds
        if i_min < 0 or i_max >= H or j_min < 0 or j_max >= W:
            raise ValueError(f"Crop bounds out of image: half_crop={half_crop}, center=({center_i}, {center_j}), radius={radius:.1f}, image_size=({H}, {W}), bounds=({i_min}, {i_max}, {j_min}, {j_max})")
        
        # Crop image
        img_cropped = img_np[i_min:i_max+1, j_min:j_max+1]
        
        # Create figure
        plt.figure(figsize=(12, 12))
        plt.imshow(img_cropped)
        
        # Draw center as small green circle (relative to cropped image)
        center_j_crop = center_j - j_min
        center_i_crop = center_i - i_min
        plt.gca().add_patch(plt.Circle((center_j_crop, center_i_crop), DEBUG_RADIUS_PX, color="green", fill=True))
        
        # Draw 36 equally spaced green pixels on the circle border
        n_points = 36
        for k in range(n_points):
            angle = 2 * math.pi * k / n_points
            border_j = center_j_crop + radius * math.cos(angle)
            border_i = center_i_crop + radius * math.sin(angle)
            border_j_int = int(round(border_j))
            border_i_int = int(round(border_i))
            # Draw single green pixel
            plt.plot(border_j_int, border_i_int, 'g.', markersize=1)
        
        plt.title(f"Moon center: (i={center_i}, j={center_j}), radius: {radius:.1f}")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
    return (center_i, center_j, radius)


class ApproxMoonFinder:
    """
    Approximate moon finder using circle edge detection.
    Precomputes circle kernels for efficient processing.
    """
    # Class attributes for precomputed kernels
    _kernels = {}  # Dict mapping radius -> kernel tensor
    _min_radius = 3
    _target_size = 256
    _max_radius = _target_size // 2 - 3
    
    @classmethod
    def _create_circle_kernel(cls, radius: int, device: torch.device) -> torch.Tensor:
        """
        Create a circle kernel with 1px thick border using distance-based approach.
        Returns kernel of shape (1, 1, kernel_size, kernel_size) on specified device.
        """
        kernel_size = 2 * radius + 1
        center = radius
        
        # Create coordinate grids
        y = torch.arange(kernel_size, dtype=torch.float32, device=device)
        x = torch.arange(kernel_size, dtype=torch.float32, device=device)
        yy, xx = torch.meshgrid(y, x, indexing='ij')
        
        # Compute distance from center
        dist = torch.sqrt((yy - center) ** 2 + (xx - center) ** 2)
        
        # Set to 1 if distance is within 0.5 of radius (1px thick border)
        kernel = (torch.abs(dist - radius) < 0.5).float()
        
        # Reshape for conv2d: (1, 1, H, W)
        return kernel.unsqueeze(0).unsqueeze(0)
    
    @classmethod
    def _get_kernel(cls, radius: int, device: torch.device) -> torch.Tensor:
        """Get or create kernel for given radius."""
        if radius not in cls._kernels:
            cls._kernels[radius] = cls._create_circle_kernel(radius, device)
        # Move kernel to requested device if needed
        kernel = cls._kernels[radius]
        if kernel.device != device:
            kernel = kernel.to(device)
            cls._kernels[radius] = kernel
        return kernel
    
    @classmethod
    def find_moon_approx(cls, img: torch.Tensor) -> tuple[int, int]:
        """
        Find moon center using approximate circle edge detection.
        img: (H, W, 3) float32 [0,1] on GPU. Returns (i, j) as tuple of ints in original image space.
        """
        assert img.ndim == 3 and img.shape[2] == 3
        original_H, original_W = img.shape[0], img.shape[1]
        device = img.device
        
        # Grayscale and downscale preserving aspect ratio, then pad/crop to 512x512
        gray = img.mean(dim=2)  # (H, W)
        
        # Compute downscaled dimensions preserving aspect ratio
        # Scale factor is min(target_size / original_size) to fit within target_size
        scale = min(cls._target_size / original_H, cls._target_size / original_W)
        H_scaled = int(round(original_H * scale))
        W_scaled = int(round(original_W * scale))
        
        # Downscale preserving aspect ratio
        gray_4d = gray.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W) for interpolation
        gray_scaled = torch.nn.functional.interpolate(
            gray_4d, size=(H_scaled, W_scaled), 
            mode='bilinear', align_corners=False
        ).squeeze()  # (H_scaled, W_scaled)
        
        # Pad to 512x512 (centered)
        pad_h = (cls._target_size - H_scaled) // 2
        pad_w = (cls._target_size - W_scaled) // 2
        gray_downscaled = torch.nn.functional.pad(
            gray_scaled, 
            (pad_w, cls._target_size - W_scaled - pad_w, pad_h, cls._target_size - H_scaled - pad_h),
            mode='constant', value=0.0
        )  # (512, 512)
        
        H_down, W_down = gray_downscaled.shape
        assert H_down == cls._target_size and W_down == cls._target_size
        
        # Initialize tracking variables
        best_diff = torch.full((H_down, W_down), float('-inf'), device=device, dtype=torch.float32)
        sum_prev = None  # Sum from 1 iteration ago
        sum_prev2 = None  # Sum from 2 iterations ago
        
        # Iterate over radii from min_radius to max_radius
        for radius in range(cls._min_radius, cls._max_radius + 1):
            # Get kernel for this radius
            kernel = cls._get_kernel(radius, device)
            
            # Each kernel needs padding equal to its radius to produce 512x512 output
            # This ensures: output_size = 512 + 2*radius - (2*radius+1) + 1 = 512
            # and all outputs are properly aligned (each pixel corresponds to same input location)
            padding = radius
            
            # Compute sum along circle using conv2d with per-kernel padding
            gray_input = gray_downscaled.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
            sum_along_circle = torch.nn.functional.conv2d(
                gray_input, kernel, padding=padding
            ).squeeze()  # (H, W)
            
            # Verify output size is correct (should always be 512x512)
            assert sum_along_circle.shape == (H_down, W_down), \
                f"Output shape mismatch: expected ({H_down}, {W_down}), got {sum_along_circle.shape} for radius {radius}"
            
            # If we have sum from 2 iterations ago, compute diff
            if sum_prev2 is not None:
                # diff = sum_now - sum_2iter_before_now
                diff = sum_along_circle - sum_prev2
                # Update best diff per pixel
                best_diff = torch.maximum(best_diff, diff)
            
            # Update history: shift by one iteration
            sum_prev2 = sum_prev
            sum_prev = sum_along_circle
        
        # Find pixel with maximum diff
        flat_idx = best_diff.argmax().item()
        i_down = flat_idx // W_down
        j_down = flat_idx % W_down
        
        # Map coordinates back to original image space
        # First, subtract padding offsets to get coordinates in scaled (non-padded) space
        i_scaled = i_down - pad_h
        j_scaled = j_down - pad_w
        
        # Then scale back to original image space
        # With align_corners=False, the mapping uses half-pixel alignment:
        # output_pos = (input_pos + 0.5) * (output_size / input_size) - 0.5
        # Inverse: input_pos = (output_pos + 0.5) * (input_size / output_size) - 0.5
        i = (i_scaled + 0.5) * (original_H / H_scaled) - 0.5
        j = (j_scaled + 0.5) * (original_W / W_scaled) - 0.5
        i = int(round(i))
        j = int(round(j))
        
        if False:
            # Visualize result
            img_np = img.cpu().numpy()
            plt.figure(figsize=(12, 8))
            plt.imshow(img_np)
            plt.gca().add_patch(plt.Circle((j, i), DEBUG_RADIUS_PX, color="green", fill=True))
            plt.title(f"Moon center (approx): (i={i}, j={j})")
            plt.axis("off")
            plt.tight_layout()
            plt.show()
        
        return (i, j)
    

In [6]:
image_infos = get_image_infos()
for ii in tqdm.tqdm(image_infos, desc="Finding moon"):
    img = Image.open(ii.path)
    img_arr = torch.from_numpy(np.array(img).astype(np.float32) / 255.0).cuda()
    i0, j0 = ApproxMoonFinder.find_moon_approx(img_arr)
    i, j, radius = find_moon(img_arr, i0, j0)
    print(ii.path, i, j, radius, ii.timestamp, ii.exposure_time)
print(len(image_infos))

Finding moon:   1%|          | 1/84 [00:01<01:36,  1.17s/it]

/home/slavik/e202602_eclipse/data/img_0150_53657076894_o.jpg 1968.1844002743917 2882.6022872059752 316.0388540762389 1712590079.0 0.00025


Finding moon:   2%|▏         | 2/84 [00:01<01:14,  1.10it/s]

/home/slavik/e202602_eclipse/data/img_0145_53657190740_o.jpg 1975.9099805227097 2871.538320876018 316.1038388195717 1712590035.0 0.00025


Finding moon:   4%|▎         | 3/84 [00:02<01:06,  1.21it/s]

...

/home/slavik/e202602_eclipse/data/img_0222_53657074589_o.jpg 1981.5159970884895 2815.7984063116182 1717.9704175608956 1712590208.0 2.0


Finding moon:  99%|█████████▉| 83/84 [01:01<00:00,  1.32it/s]

/home/slavik/e202602_eclipse/data/img_0223_53656722276_o.jpg 2031.9409763797416 3037.5976801750944 1798.7583265560543 1712590211.0 2.0


Finding moon: 100%|██████████| 84/84 [01:02<00:00,  1.35it/s]

/home/slavik/e202602_eclipse/data/img_0227_53655846522_o.jpg 2160.814335231319 2745.7094268657816 2075.5121771642607 1712590225.0 1.0
84


In [7]:
# Bad news about sun / moon apparent motion
# https://chatgpt.com/share/e/6996a405-7df4-800e-9c16-51b51a28ce9e

# if solar radius is 500px, then:
# 1px ~ 1.92 arcsec
# scene as a whole will move by 2344 px in 5 minutes (becays its 15deg in hour, so in pixels and in 5min we have (15*3600/1.92) * (5/60))
# moon apparent movement wrt sun: 80px in 5min
# moon radius minus sun radius: maximally 40px

In [8]:
if False:
    # Two-image semitransparent debug: same crop region (from first image), both overlaid
    PATH1 = "/home/slavik/e202602_eclipse/data/img_0148_53656946138_o.jpg"
    PATH2 = "/home/slavik/e202602_eclipse/data/img_0149_53655849382_o.jpg"
    
    img1 = Image.open(PATH1)
    img2 = Image.open(PATH2)
    arr1 = np.array(img1).astype(np.float32) / 255.0
    arr2 = np.array(img2).astype(np.float32) / 255.0
    H, W = arr1.shape[0], arr1.shape[1]
    
    t1 = torch.from_numpy(arr1).cuda()
    t2 = torch.from_numpy(arr2).cuda()
    i1, j1, r1 = find_moon(t1, H / 2, W / 2)
    i2, j2, r2 = find_moon(t2, H / 2, W / 2)
    
    # Crop region from first image (same as in find_moon debug)
    half_crop = int(round(1.1 * r1))
    i_min = int(round(i1 - half_crop))
    i_max = int(round(i1 + half_crop))
    j_min = int(round(j1 - half_crop))
    j_max = int(round(j1 + half_crop))
    # Clamp to image bounds so same region works for both
    i_min = max(0, i_min)
    i_max = min(H - 1, i_max)
    j_min = max(0, j_min)
    j_max = min(W - 1, j_max)

    crop1 = arr1[i_min : i_max + 1, j_min : j_max + 1]
    crop2 = arr2[i_min : i_max + 1, j_min : j_max + 1]
    blended = 0.5 * crop1 + 0.5 * crop2
    
    # Draw same debug as find_moon: green circle at center + 36 border points per moon
    plt.figure(figsize=(12, 12))
    plt.imshow(blended)
    
    # Moon 1 (first image) in crop coords
    c1_j = j1 - j_min
    c1_i = i1 - i_min
    plt.gca().add_patch(plt.Circle((c1_j, c1_i), DEBUG_RADIUS_PX, color="green", fill=True))
    for k in range(36):
        angle = 2 * math.pi * k / 36
        bj = c1_j + r1 * math.cos(angle)
        bi = c1_i + r1 * math.sin(angle)
        plt.plot(bj, bi, "g.", markersize=1)
    
    # Moon 2 (second image) in crop coords
    c2_j = j2 - j_min
    c2_i = i2 - i_min
    plt.gca().add_patch(plt.Circle((c2_j, c2_i), DEBUG_RADIUS_PX, color="green", fill=True))
    for k in range(36):
        angle = 2 * math.pi * k / 36
        bj = c2_j + r2 * math.cos(angle)
        bi = c2_i + r2 * math.sin(angle)
        plt.plot(bj, bi, "g.", markersize=1)
    
    plt.title("Two images overlaid (same crop); green = moon centers and radii")
    plt.axis("off")
    plt.tight_layout()
    plt.show()